In [1]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
EPISODE_PATH = Path("~/Projects/mimic-video/data/episode_0057.zarr").expanduser()

keys = [p.name for p in sorted(EPISODE_PATH.iterdir()) if not p.name.startswith('.')]
arrays = {k: zarr.open_array(str(EPISODE_PATH / k), 'r') for k in keys}

for k, arr in arrays.items():
    print(f"{k:45s} shape={str(arr.shape):25s} dtype={arr.dtype}")

joint_action_lowdim                           shape=(625, 14)                 dtype=float32
joint_action_lowdim_timestamps                shape=(625,)                    dtype=uint64
joint_state_lowdim                            shape=(625, 14)                 dtype=float32
joint_state_lowdim_timestamps                 shape=(625,)                    dtype=uint64
language_embedding                            shape=(1, 512, 1024)            dtype=float16
language_embedding_timestamps                 shape=(1,)                      dtype=uint64
language_instruction                          shape=(1,)                      dtype=object
language_instruction_timestamps               shape=(1,)                      dtype=uint64
workspace_rgb                                 shape=(586, 480, 640, 3)        dtype=uint8
workspace_rgb_timestamps                      shape=(625,)                    dtype=uint64
wrist_rgb_left                                shape=(625, 480, 640, 3)        dtype=uint

In [ ]:
lang = arrays['language_instruction'][:]
print("Task:", lang[0].decode() if isinstance(lang[0], bytes) else lang[0])

In [ ]:
joint_state = arrays['joint_state_lowdim'][:]
joint_action = arrays['joint_action_lowdim'][:]
T = joint_state.shape[0]
n_joints = joint_state.shape[1]

fig, axes = plt.subplots(n_joints, 1, figsize=(14, n_joints * 1.2), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(joint_state[:, i], label='state', linewidth=1)
    ax.plot(joint_action[:, i], label='action', linewidth=1, linestyle='--', alpha=0.7)
    ax.set_ylabel(f'j{i}', fontsize=8)
    ax.tick_params(labelsize=7)
axes[0].legend(fontsize=8)
axes[-1].set_xlabel('timestep')
fig.suptitle(f'Joint state vs action  —  T={T}', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
rgb = arrays['workspace_rgb']
n_frames = 6
idxs = np.linspace(0, T - 1, n_frames, dtype=int)

fig, axes = plt.subplots(1, n_frames, figsize=(3 * n_frames, 3))
for ax, idx in zip(axes, idxs):
    ax.imshow(rgb[idx])
    ax.set_title(f't={idx}', fontsize=8)
    ax.axis('off')
fig.suptitle('workspace_rgb', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
for cam in ('wrist_rgb_left', 'wrist_rgb_right'):
    if cam not in arrays:
        print(f"{cam} not present")
        continue
    wrist = arrays[cam]
    fig, axes = plt.subplots(1, n_frames, figsize=(3 * n_frames, 3))
    for ax, idx in zip(axes, idxs):
        ax.imshow(wrist[idx])
        ax.set_title(f't={idx}', fontsize=8)
        ax.axis('off')
    fig.suptitle(cam, fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
delta = joint_action - joint_state
fig, ax = plt.subplots(figsize=(14, 4))
for i in range(n_joints):
    ax.plot(delta[:, i], label=f'j{i}', linewidth=1)
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_xlabel('timestep')
ax.set_ylabel('action - state')
ax.set_title('Action residual per joint')
ax.legend(ncol=7, fontsize=8)
plt.tight_layout()
plt.show()